# Mountain Named Entity Recognition
## Dataset Creation

This notebook demonstrates the process of creating a synthetic dataset for the Mountain Named Entity Recognition (NER) task.

In [1]:
import re
import random
import requests
import numpy as np
import pandas as pd

from io import StringIO
from datasets import Dataset, DatasetDict

## Collecting mountain names from Wikipedia.

Mountain names were automatically collected from the Wikipedia page List of mountains by topographic prominence. Using an external source makes the dataset reproducible and avoids maintaining a manually created list.

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_mountain_peaks_by_prominence"

headers = {"User-Agent": "Mozilla/5.0"}

# Set a browser-like User-Agent to avoid request restrictions
html = requests.get(url, headers=headers).text

tables = pd.read_html(html)

print(len(tables))

2


/tmp/ipykernel_1531/2149617595.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(html)


Wikipedia pages often contain multiple HTML tables. To identify the table containing
the mountain names, all extracted tables are briefly inspected. The first table
contains the Peak column, which is used as the source of mountain names.

In [ ]:
for i, table in enumerate(tables):
    print(i)
    print(table.head())

0
   No.                      Peak             Range (or island)       Location  \
0  1.0             Mount Everest                     Himalayas    China Nepal   
1  2.0                 Aconcagua                         Andes      Argentina   
2  3.0  Denali / Mount McKinley*                  Alaska Range  United States   
3  4.0        Mount Kilimanjaro*        Eastern Rift mountains       Tanzania   
4  5.0        Pico Simón Bolívar  Sierra Nevada de Santa Marta       Colombia   

                                   Coordinates [1]  Prominence (m)  \
0     27°59′17″N 86°55′30″E﻿ / ﻿27.9881°N 86.925°E         8848.86   
1    32°39′11″S 70°00′42″W﻿ / ﻿32.6531°S 70.0117°W         6960.80   
2  63°04′09″N 151°00′23″W﻿ / ﻿63.0692°N 151.0064°W         6155.00   
3      3°04′00″S 37°21′33″E﻿ / ﻿3.0667°S 37.3592°E         5885.00   
4    10°50′18″N 73°41′12″W﻿ / ﻿10.8383°N 73.6867°W         5529.00   

   Height (m)  Col (m) Encirclement parent Prominence parent  
0     8848.86        0     

The Peak column was selected because it contains the official names of mountain peaks, while the remaining columns provide additional metadata such as elevation, prominence and geographic information.

In [ ]:
mountains = tables[0]["Peak"].dropna().tolist()

## Cleaning and normalizing the extracted names.

In [ ]:
print(mountains[:20])
print(len(mountains))

['Mount Everest', 'Aconcagua', 'Denali / Mount McKinley*', 'Mount Kilimanjaro*', 'Pico Simón Bolívar', 'Mount Logan', 'Pico de Orizaba', 'Vinson Massif', 'Puncak Jaya', 'Mount Elbrus', 'Monte Bianco/Mont Blanc', 'Mount Damavand', 'Klyuchevskaya Sopka', 'Nanga Parbat', 'Mauna Kea', 'Jengish Chokusu', 'Bogda Peak', 'Chimborazo', 'Namcha Barwa', 'Mount Kinabalu']
125


Add a missing mountain that was not included in the extracted Wikipedia table.

In [ ]:
mountains.append('Broad Peak')

Clean extracted mountain names by:
 - removing citation markers (e.g. [1])
 - removing asterisks
 - trimming whitespace
 - splitting alternative names

In [ ]:
clean_mountains = []

for mountain in mountains:

    mountain = re.sub(r"\[.*?\]", "", mountain)  # remove [1]
    mountain = mountain.replace("*", "")         # remove *
    mountain = mountain.strip()

    if "/" in mountain:
        clean_mountains.extend([part.strip() for part in mountain.split("/")])
    else:
        clean_mountains.append(mountain)

Due to a GitHub notebook rendering issue, the regular expression in the third line of the code cell may be displayed incorrectly.
The correct line should be:
```mountain = re.sub(r"\[.*?\]", "", mountain)```

Remove duplicates and sort mountain names alphabetically.

In [ ]:
mountains = sorted(set(clean_mountains))

In [ ]:
print(f"Number of mountains: {len(mountains)}")
print(mountains[:5])

Number of mountains: 129
['Aconcagua', 'Aneto', 'Annapurna', 'Aoraki', 'Arjuno-Welirang']


Mountain names may appear in text in different forms. For example, *Mount Everest* is often shortened to *Everest*. To improve annotation quality and help the model generalize better, alternative aliases are generated automatically.

Some mountains are excluded from this process because removing the prefix *Mount* would produce ambiguous names (e.g. Cook, Kenya, Temple) that may refer
to people, places or common nouns rather than mountains.

In [ ]:
def build_aliases(mountains):
    """
    Generate alternative names for each mountain.

    Examples:
        Mount Everest -> ["Mount Everest", "Everest"]
        Mont Blanc    -> ["Mont Blanc", "Blanc"]

    These aliases increase the diversity of the generated dataset by allowing
    mountain names to appear both with and without common prefixes.
    """

    exceptions = {
        "Mount Kenya",
        "Mount Cook",
        "Mount Graham",
        "Mount Temple",
        "Mount Adams",
        "Mount Mitchell",
        "Mount Massive",
    }

    aliases = {}

    for mountain in mountains:

        names = [mountain]

        if mountain not in exceptions:

            if mountain.startswith("Mount "):
                names.append(mountain.replace("Mount ", "", 1))

            elif mountain.startswith("Mont "):
                names.append(mountain.replace("Mont ", "", 1))

        aliases[mountain] = list(dict.fromkeys(names))

    return aliases

In [ ]:
aliases = build_aliases(mountains)

Display several examples to verify the generated aliases.

In [ ]:
print(aliases["Mount Everest"])
print(aliases["Mount Etna"])
print(aliases["Mount Kenya"])

['Mount Everest', 'Everest']
['Mount Etna', 'Etna']
['Mount Kenya']


## Generating synthetic sentences.

Since no publicly available dataset of sentences containing mountain-related named entities was available, a synthetic corpus was generated using ChatGPT. The generated sentences were designed to imitate realistic text while providing sufficient diversity for training an NER model.

Generation requirements:

- Cover topics related to mountains, hiking, geography, travel, nature, weather, expeditions and tourism
- Include sentences containing one mountain name
- Include sentences containing multiple mountain names
- Include approximately 20-30% negative examples containing no mountain names
- Mix different writing styles:
    - conversational
    - news
    - travel blogs
    - educational
    - historical facts
    - weather reports
- Vary sentence length
- Use only real mountain names
- Output one sentence per line

In [ ]:
sentences = [
    "Everest appeared above the morning clouds just as the first rays of sunlight reached the valley.",
"We left the campsite before dawn to avoid hiking during the hottest part of the day.",
"K2 has challenged experienced climbers with its steep faces and unpredictable weather for decades.",
"The trail wound through a quiet forest filled with birds and wildflowers.",
"Tourists often compare the views from Matterhorn and Mont Blanc during their Alpine holidays.",
"A gentle breeze made the afternoon trek much more enjoyable.",
"Denali dominates the surrounding wilderness with remarkable prominence.",
"The guide explained how glaciers slowly carve valleys over thousands of years.",
"Many photographers visit Kilimanjaro during the dry season for clear summit views.",
"Our backpacks felt lighter after we stopped for lunch beside the river.",
"Annapurna attracts trekkers from around the world with its spectacular scenery.",
"The local forecast predicted scattered showers in the higher elevations.",
"Aconcagua remained visible long after the lower hills disappeared behind us.",
"The wooden bridge crossed a fast-moving mountain stream.",
"Everest and Lhotse share one of the world's most famous skylines.",
"The ranger reminded everyone to stay on the marked trail.",
"Fuji looked perfectly symmetrical beneath the clear blue sky.",
"Several hikers paused to photograph colorful alpine flowers.",
"Makalu rises not far from Everest, creating an unforgettable Himalayan landscape.",
"The weather changed suddenly as dark clouds gathered overhead.",
"Ben Nevis welcomed visitors with surprisingly calm conditions that morning.",
"The valley echoed with the sound of rushing water.",
"Kangchenjunga remained hidden behind thick clouds until late afternoon.",
"Our guide pointed out several native plants growing beside the path.",
"Cho Oyu is frequently chosen by climbers preparing for even higher peaks.",
"The mountain lodge served warm soup after a cold day's hike.",
"Dhaulagiri stood brilliantly against the fresh snow covering the surrounding ridges.",
"The sunset painted the western sky with shades of orange and pink.",
"Rain arrived just as we reached the viewpoint overlooking the valley.",
"Manaslu has become increasingly popular among experienced expedition teams.",
"The path climbed steadily through dense pine forests.",
"Ama Dablam is admired for its elegant profile and dramatic setting.",
"The map highlighted nearby rivers, forests, and protected natural areas.",
"Travelers often visit both Rainier and Mount St. Helens during the same trip.",
"We spotted a family of deer grazing near the edge of the meadow.",
"Elbrus remains snow-covered for much of the year.",
"The campsite offered beautiful views of the surrounding hills.",
"The route between Matterhorn and Weisshorn rewards hikers with stunning Alpine scenery.",
"The cool morning air made the climb much easier.",
"Cook rises above New Zealand's Southern Alps with impressive beauty.",
"Several visitors gathered outside to watch the sunrise.",
"Toubkal attracts hikers seeking North Africa's highest summit.",
"The guide checked everyone's equipment before continuing the ascent.",
"Table Mountain overlooks Cape Town with dramatic cliffs on every side.",
"Heavy fog reduced visibility along the upper section of the trail.",
"Fuji attracts millions of visitors during the official climbing season.",
"A small waterfall flowed beside the narrow hiking path.",
"Everest, Lhotse, and Nuptse formed an incredible backdrop for our trek.",
"The weather remained clear throughout the afternoon.",
"Shasta is easily recognized by its snow-covered volcanic cone.",
"We reached the viewpoint just before sunset.",
"Kosciuszko offers accessible hiking routes suitable for many visitors.",
"The air became noticeably cooler as we gained elevation.",
"Rainier supports numerous glaciers despite its relatively southern location.",
"The travel blog recommended arriving early to avoid large crowds.",
"Merapi remains one of Indonesia's most active volcanoes.",
"Fresh snow covered the highest ridges overnight.",
"Whitney provides sweeping views across the Sierra Nevada.",
"Our group stopped frequently to take photographs.",
"The skyline looked especially dramatic with Denali in the distance.",
"Local guides shared stories about the region's wildlife and geology.",
"Teide rises above the Canary Islands with a striking volcanic landscape.",
"The forest became quieter as evening approached.",
"Ararat dominates the surrounding plains on clear days.",
"A light rain cooled the valley during the afternoon.",
"Kenya features several distinct ecological zones from forest to alpine terrain.",
"The rocky trail demanded careful footing after the storm.",
"Travelers admired both Elbrus and Kazbek during their journey through the Caucasus.",
"The mountain air felt crisp and refreshing.",
"Kinabalu is famous for its exceptional biodiversity.",
"Our campsite overlooked a peaceful alpine lake.",
"Logan is the highest mountain in Canada by elevation.",
"The clouds slowly drifted away before noon.",
"Baker and Glacier Peak often receive heavy snowfall during winter.",
"The winding road climbed steadily toward the national park entrance.",
"The guide encouraged everyone to refill their water bottles.",
"Popocatépetl continues to be monitored because of its volcanic activity.",
"Colorful wildflowers lined both sides of the hiking trail.",
"Eiger, Mönch, and Jungfrau create one of Europe's most iconic mountain panoramas.",
"The forecast predicted colder temperatures above two thousand meters.",
"Rinjani surrounds visitors with impressive volcanic scenery and a beautiful crater lake.",
"The peaceful atmosphere made the long walk feel effortless.",
"Many travelers combine visits to Bromo and Semeru while exploring Java.",
"The sunrise reflected beautifully across the calm lake.",
"Damavand appears frequently in Persian literature and folklore.",
"Our guide recommended hiking earlier to avoid afternoon thunderstorms.",
"Nanga Parbat has earned a reputation for difficult climbing conditions.",
"The trail crossed several wooden bridges before reaching the valley.",
"Snowdon attracts walkers throughout every season.",
"The river carved deep channels through the rocky landscape.",
"Visitors often photograph Ama Dablam from nearby trekking routes.",
"A cool wind swept across the open ridge.",
"Pelée dramatically changed the surrounding region during its historic eruption.",
"The campsite remained quiet throughout the night.",
"Robson rises above the Canadian Rockies with remarkable prominence.",
"Morning mist slowly disappeared as the temperature increased.",
"Travelers compared the landscapes around Aconcagua and Ojos del Salado before planning their expedition.",
"The final stretch of the trail rewarded us with unforgettable panoramic views.",
"Everest disappeared behind drifting clouds just before sunset.",
"The narrow trail followed the edge of a deep valley.",
"K2 and Gasherbrum I dominate the remote landscape of the Karakoram.",
"A cool breeze carried the scent of pine through the forest.",
"Fuji reflected beautifully in the calm waters of the nearby lake.",
"The hiking route passed several waterfalls before reaching the campsite.",
"Mont Blanc attracts climbers, photographers, and skiers throughout the year.",
"The valley was covered in colorful wildflowers after weeks of rain.",
"Denali remained hidden until the morning fog finally cleared.",
"Our guide reminded everyone to pack enough drinking water.",
"Matterhorn rises dramatically above the surrounding Alpine villages.",
"The forecast called for clear skies and light winds tomorrow.",
"Kilimanjaro offers several routes with different levels of difficulty.",
"We rested beside a quiet stream before continuing uphill.",
"Everest and Makalu looked spectacular beneath the bright afternoon sun.",
"The wooden observation deck overlooked a beautiful glacier.",
"Annapurna is surrounded by traditional villages and scenic trekking trails.",
"Clouds gathered quickly over the western ridge.",
"Cho Oyu is often chosen as preparation for higher Himalayan expeditions.",
"The campsite became lively as more hikers arrived before dusk.",
"Ben Nevis experiences rapidly changing weather throughout the year.",
"The map showed several alternative hiking routes.",
"Aconcagua remained visible for hours across the open plains.",
"Fresh snow covered the highest peaks overnight.",
"Lhotse stands immediately south of Everest.",
"We enjoyed a warm meal after a long day on the trail.",
"Dhaulagiri and Annapurna are separated by one of the world's deepest gorges.",
"The morning air felt cool and refreshing.",
"Elbrus attracts climbers from across Europe every summer.",
"The forest trail became quieter as evening approached.",
"Rainier supports one of the largest glacier systems in the contiguous United States.",
"The weather improved shortly after sunrise.",
"Kangchenjunga remained hidden behind thick clouds for most of the morning.",
"The rocky path required careful footing after the rain.",
"Cook dominates New Zealand's Southern Alps with impressive glaciers.",
"Several hikers stopped to photograph a rainbow above the valley.",
"Rinjani features a spectacular crater lake near its summit.",
"The local guide shared fascinating stories about the area's history.",
"Toubkal welcomes thousands of trekkers each year.",
"A gentle breeze kept temperatures comfortable during the climb.",
"Shasta stood out clearly against the cloudless sky.",
"The trail crossed open meadows filled with blooming flowers.",
"Merapi continues to shape the surrounding landscape through volcanic activity.",
"Our backpacks felt much heavier near the end of the hike.",
"Whitney attracts experienced hikers seeking a challenging adventure.",
"The sound of rushing water echoed through the canyon.",
"Everest, Nuptse, and Lhotse formed a breathtaking panorama.",
"The travel magazine recommended visiting during early autumn.",
"Kosciuszko offers some of Australia's most accessible alpine walks.",
"The valley remained peaceful despite the growing number of visitors.",
"Table Mountain was covered by a blanket of low clouds.",
"The guide explained how erosion creates dramatic mountain scenery.",
"Nanga Parbat has fascinated mountaineers for generations.",
"Strong winds forced several groups to delay their ascent.",
"Damavand rises prominently above the surrounding Iranian plateau.",
"The sunrise painted the snowy slopes in golden light.",
"Baker and Rainier are both famous volcanic peaks in the Pacific Northwest.",
"We spotted several mountain goats near the rocky cliffs.",
"Eiger presents technical climbing routes admired by experienced alpinists.",
"The forecast warned of possible thunderstorms later in the afternoon.",
"Kazbek towers above picturesque valleys in the Caucasus.",
"Our campsite overlooked a crystal-clear alpine lake.",
"Ama Dablam is considered one of the most beautiful peaks in the Himalayas.",
"The trail gradually climbed through dense evergreen forests.",
"Popocatépetl remained visible despite the distant haze.",
"The local visitor center offered free hiking maps.",
"Robson is the tallest peak in the Canadian Rockies.",
"The evening sky turned bright orange before darkness arrived.",
"Bromo and Semeru create unforgettable volcanic landscapes on Java.",
"The path narrowed as it approached the rocky ridge.",
"Teide dominates the skyline of Tenerife.",
"We paused often to admire the surrounding scenery.",
"Manaslu has become increasingly popular with international expeditions.",
"The weather station reported falling temperatures overnight.",
"Glacier Peak remains one of the least visited volcanoes in Washington.",
"The quiet valley echoed with birdsong at dawn.",
"Fuji inspired countless paintings and poems throughout Japanese history.",
"A wooden bridge crossed the icy river below.",
"Logan rises above vast wilderness in northwestern Canada.",
"The group celebrated after reaching the scenic overlook.",
"Kinabalu supports an extraordinary variety of plant species.",
"The fresh mountain air was incredibly refreshing.",
"Ararat remained snow-covered despite the warm conditions below.",
"The guide encouraged everyone to leave no trace.",
"Pelée dramatically changed the history of Martinique.",
"The winding road revealed breathtaking views around every corner.",
"Snowdon offers rewarding walks for hikers of all experience levels.",
"Cool mist drifted through the pine forest.",
"Everest attracts climbers from nearly every continent.",
"The route became steeper after passing the old stone shelter.",
"Elgon stretches across the border between Kenya and Uganda.",
"Our travel group arrived just before sunset.",
"Ojos del Salado is the world's highest active volcano.",
"The peaceful landscape encouraged us to stay longer.",
"Triglav symbolizes Slovenia's natural heritage.",
"The clouds finally cleared after several hours of rain.",
"Assiniboine is often called the Matterhorn of the Canadian Rockies.",
"The trail ended beside a beautiful glacial lake.",
"Makalu and Kangchenjunga stand among the highest mountains on Earth.",
"The cool evening breeze marked the end of another unforgettable day.",
"Everest remained visible long after the surrounding hills disappeared into the clouds.",
"The trail passed through a peaceful forest alive with birdsong.",
"K2 stands as one of the world's most demanding climbing destinations.",
"A light drizzle made the rocky path slightly slippery.",
"Matterhorn and Dent Blanche create a spectacular skyline in the Swiss Alps.",
"The guide suggested taking frequent breaks during the ascent.",
"Kilimanjaro offers hikers an incredible journey through several climate zones.",
"Our campsite overlooked a quiet valley surrounded by pine trees.",
"Annapurna is famous for both its beauty and its challenging terrain.",
"The forecast predicted sunny skies with cooler temperatures at higher elevations.",
"Denali rises dramatically above the Alaskan wilderness.",
"We reached the summit just before the afternoon clouds arrived.",
"Mont Blanc remains one of Europe's most visited alpine destinations.",
"The valley echoed with the sound of distant waterfalls.",
"Everest and Cho Oyu belong to the vast Himalayan mountain system.",
"A gentle breeze carried the scent of fresh grass across the meadow.",
"Fuji looked stunning beneath the clear evening sky.",
"The hiking trail followed an old stone pathway through the hills.",
"Lhotse shares part of its massive ridge with Everest.",
"The visitor center displayed detailed maps of nearby trails.",
"Ben Nevis welcomed us with surprisingly calm weather.",
"Heavy clouds gathered over the western horizon.",
"Makalu impressed everyone with its sharp pyramid-shaped summit.",
"The local café served hot tea after the chilly hike.",
"Rainier remains one of the most heavily glaciated peaks in the United States.",
"We spotted several eagles circling above the cliffs.",
"Dhaulagiri dominates the western Nepalese landscape.",
"The wooden bridge crossed a fast-flowing mountain river.",
"Elbrus attracts visitors throughout both summer and winter.",
"The air became noticeably thinner as we continued climbing.",
"Table Mountain provides panoramic views of Cape Town and the Atlantic Ocean.",
"Several travelers stopped to photograph the colorful sunset.",
"Cook and Tasman dominate New Zealand's Southern Alps.",
"Our guide shared fascinating geological facts during the trek.",
"Toubkal is a favorite destination for trekkers exploring Morocco.",
"The weather remained stable for the entire expedition.",
"Aconcagua rises high above the surrounding Andes.",
"The forest became quieter as evening approached.",
"Merapi continues to remind nearby communities of nature's power.",
"The narrow path climbed steadily toward the ridge.",
"Whitney rewards hikers with spectacular views across the Sierra Nevada.",
"The morning fog slowly disappeared after sunrise.",
"Eiger, Mönch, and Jungfrau attract climbers from around the globe.",
"We paused to refill our water bottles beside a spring.",
"Kinabalu is home to thousands of unique plant species.",
"The clear night sky revealed countless stars above the campsite.",
"Kangchenjunga remained partially hidden behind drifting clouds.",
"The guide recommended wearing waterproof boots after recent rainfall.",
"Shasta appeared especially beautiful after the overnight snowfall.",
"The winding trail offered unforgettable valley views.",
"Popocatépetl is closely monitored because of ongoing volcanic activity.",
"A cool wind swept across the exposed ridge.",
"Ama Dablam is admired for its elegant silhouette.",
"The journey became easier once the steep climb ended.",
"Baker and Glacier Peak often receive significant snowfall each winter.",
"The surrounding meadows were covered with blooming wildflowers.",
"Fuji attracts millions of visitors every year.",
"The observation platform overlooked a peaceful alpine lake.",
"Kosciuszko provides gentle hiking opportunities for families.",
"The travel blog recommended arriving before sunrise.",
"Ararat dominates the horizon on clear autumn mornings.",
"The rocky slope required careful footing throughout the climb.",
"Semeru is the tallest mountain on the island of Java.",
"The afternoon sunshine warmed the valley floor.",
"Everest, Makalu, and Lhotse formed an unforgettable Himalayan panorama.",
"Our group celebrated after completing the longest section of the trail.",
"Damavand appears frequently in Persian legends and literature.",
"The fresh mountain air made everyone feel energized.",
"Robson towers above the surrounding Canadian wilderness.",
"The forecast warned hikers about strong winds near the summit.",
"Nanga Parbat has challenged generations of experienced mountaineers.",
"We watched the sunrise from a quiet viewpoint overlooking the valley.",
"Bromo creates spectacular volcanic scenery before dawn.",
"The trail crossed several small wooden bridges.",
"Cho Oyu is one of the world's fourteen peaks above eight thousand meters.",
"The campsite remained peaceful throughout the night.",
"Logan is Canada's highest mountain by elevation.",
"A family of deer crossed the trail ahead of us.",
"Teide rises above Tenerife with an unmistakable volcanic profile.",
"The guide pointed out unusual rock formations beside the path.",
"Manaslu attracts climbers seeking quieter Himalayan expeditions.",
"The river flowed swiftly through the narrow canyon.",
"Elgon contains one of the largest volcanic calderas in the world.",
"Birdsong filled the forest throughout the early morning.",
"Triglav is deeply connected to Slovenia's national identity.",
"We packed extra food before beginning the hike.",
"Pelée transformed the surrounding landscape during its famous eruption.",
"The weather changed rapidly after midday.",
"Rinjani features a breathtaking crater lake near its summit.",
"Our hiking group enjoyed a well-earned rest beside the stream.",
"Kazbek rises above picturesque villages in the Caucasus.",
"The valley looked especially beautiful after fresh rainfall.",
"Assiniboine resembles a perfect pyramid from many viewpoints.",
"The path curved gently through fields of colorful wildflowers.",
"Everest continues to inspire explorers from around the world.",
"The final climb rewarded us with magnificent panoramic views.",
    "Many hikers dream of standing on the summit of Mount Everest at least once in their lives.",
"The trail was muddy after last night's rain, but the forest looked more beautiful than ever.",
"Mont Blanc attracts climbers from around the world every summer.",
"The sunrise over Mount Fuji painted the sky with shades of orange and pink.",
"Our guide explained how glaciers slowly reshape mountain landscapes over thousands of years.",
"Kanchenjunga remains one of the world's most challenging high peaks.",
"The forecast predicts strong winds across the alpine region this afternoon.",
"Hikers often compare the technical demands of K2 and Broad Peak.",
"We packed extra water because the weather was expected to become unusually warm.",
"The cable car offers spectacular views of Teide and the surrounding volcanic terrain.",
"Mount Kilimanjaro is famous for allowing trekkers to climb from tropical forests to icy slopes.",
"Snowfall can quickly change trail conditions even during late spring.",
"The expedition crossed remote valleys before reaching Manaslu Base Camp.",
"Tourists visiting Denali often hope to spot bears and moose along the park roads.",
"Local villagers shared stories about Mount Kenya that had been passed down for generations.",
"The geology lecture focused on how volcanic mountains are formed.",
"Climbers preparing for Dhaulagiri spend months improving their endurance.",
"Annapurna has earned a reputation for demanding extreme caution from mountaineers.",
"The hiking route offered quiet lakes, wildflowers, and breathtaking scenery.",
"Aoraki looked magnificent as fresh snow covered its highest ridges.",
"Mount Elbrus and Mont Blanc are both popular goals for European mountaineers.",
"Clouds gathered around Mount Rainier shortly before sunset.",
"Some expeditions to Nanga Parbat have been delayed by severe storms.",
"The visitor center provides maps, safety advice, and information about local wildlife.",
"Experienced climbers often train on Mount Shasta before attempting more difficult objectives.",
"Chimborazo's summit is the farthest point from Earth's center because of the planet's shape.",
"The mountain refuge served hot soup after a long day on the trail.",
"Rakaposhi rises dramatically above the surrounding valleys.",
"Mount Logan is higher than Mount Lucania, and both stand in Canada's rugged wilderness.",
"The path became steeper with every kilometer we climbed.",
"Volcanic activity at Mount Etna continues to attract scientists and curious travelers.",
"Pico de Orizaba is the tallest mountain in Mexico.",
"Morning fog reduced visibility across the high ridges.",
"Mount Agung and Mount Rinjani are well known for their volcanic landscapes.",
"A gentle breeze made the hike much more comfortable.",
"The history museum displayed photographs from early expeditions to Mount Blackburn.",
"Tirich Mir dominates the skyline of northern Pakistan.",
"Several photographers waited patiently for the clouds to clear.",
"Shishapangma is the lowest of the world's fourteen eight-thousand-meter peaks.",
"Mount Cameroon offers diverse ecosystems from rainforest to volcanic slopes.",
"The weather improved just in time for the afternoon ascent.",
"Mount Robson is widely admired for its dramatic appearance.",
"Belukha Mountain marks part of the border region between Kazakhstan and China.",
"Scientists continue to monitor Mount Erebus because of its persistent volcanic activity.",
"Our boots were soaked before we reached the first viewpoint.",
"The snowfields of Mount Cook sparkle brilliantly under clear blue skies.",
"Mount Saint Elias and Mount Fairweather rise above spectacular coastal landscapes.",
"The guide recommended starting before dawn to avoid afternoon thunderstorms.",
"Semeru frequently releases ash into the atmosphere.",
"The trail to Pico Duarte passes through beautiful pine forests.",
"Climbers often debate whether Denali or Mount Vinson presents greater logistical challenges.",
"Mount Vinson is officially known as Vinson Massif.",
"Mauna Kea offers exceptional conditions for astronomical observatories.",
"The family enjoyed a peaceful picnic beside a mountain stream.",
"Mount Damavand is the highest peak in Iran.",
"Haleakalā is famous for its vast volcanic crater and unforgettable sunrises.",
"Jade Dragon Snow Mountain overlooks the historic city of Lijiang.",
"Mount Apo is the highest mountain in the Philippines.",
"Heavy snowfall forced several hiking groups to postpone their plans.",
"Mount Kinabalu supports an extraordinary variety of plant species.",
"Puncak Jaya is the highest mountain on the island of New Guinea.",
"The documentary highlighted the biodiversity found in alpine environments.",
"Popocatepetl remains one of Mexico's most closely monitored volcanoes.",
"Gangkhar Puensum has never been officially climbed.",
"Strong winds swept across the exposed ridge throughout the afternoon.",
"Mount Ararat has long held cultural and historical significance.",
"Travelers visiting Pico do Fogo often explore nearby volcanic villages.",
"Mount Kerinci towers above the rainforests of Sumatra.",
"Klyuchevskaya Sopka is the tallest active volcano in Eurasia.",
"The campsite was surprisingly quiet after sunset.",
"Namcha Barwa stands near one of the deepest river gorges in the world.",
"Ismoil Somoni Peak was once known by several different names.",
"Mount Stanley lies within the Rwenzori mountain range.",
"The scenic drive revealed distant views of Mount Waddington.",
"Mount Whitney is the highest summit in the contiguous United States.",
"The ranger reminded everyone to stay on marked trails.",
"Nanda Devi is surrounded by a protected national park.",
"Jengish Chokusu and Pik Talgar are important peaks in Central Asia.",
"A cold front brought fresh snow to the upper elevations overnight.",
"Mount Gongga is known for its dramatic relief and unpredictable weather.",
"Kongur Tagh rises prominently above western China.",
"The travel blog described unforgettable evenings around a campfire.",
"Sabalan is famous for the crater lake near its summit.",
"Toubkal is a favorite destination for trekkers visiting Morocco.",
"Mount Meru is often climbed as preparation for Mount Kilimanjaro.",
"Shiveluch remains one of the most active volcanoes in Kamchatka.",
"Tomort features impressive glaciers and rugged alpine scenery.",
"The educational program taught children how to read topographic maps.",
"Batura Sar, Rakaposhi, and K2 are among the magnificent peaks of northern Pakistan.",
"Mount Hayes and Mount Marcus Baker dominate parts of Alaska's mountainous landscape.",
"Pico Basilé offers panoramic views across Bioko Island.",
"Mount Wilhelm is the highest mountain in Papua New Guinea.",
"The weather report warned hikers about possible lightning at higher elevations.",
"Cerro Chirripó is the highest mountain in Costa Rica.",
"Ojos del Salado is the world's highest active volcano.",
"Mount Lawu and Mount Slamet are popular hiking destinations in Indonesia.",
"Emi Koussi rises from the Sahara as a massive volcanic mountain.",
"The expedition celebrated after successfully reaching the summit despite difficult conditions.",
"Mount Leuser shelters rich tropical biodiversity.",
"Gyala Peri stands close to Namcha Barwa in the eastern Himalayas.",
"A clear autumn morning is often the best time to enjoy mountain views.",
"Mount Karisimbi is the highest volcano in the Virunga Mountains.",
"Climbers on Kamet and Nanda Devi must carefully plan for changing weather.",
"Pico da Neblina lies deep within the Amazon rainforest near Brazil's northern border."
]

In [ ]:
len(sentences)

398

## Creating BIO annotations

The generated sentences are automatically converted into the BIO tagging format,
which is commonly used for Named Entity Recognition tasks.

Tag definitions:

B-MOUNTAIN -- first token of a mountain name

I-MOUNTAIN -- continuation of a mountain name

O -- non-entity token

In [ ]:
def create_bio(sentence, aliases):

    # Tokenize the input sentence while preserving punctuation
    tokens = re.findall(r"\w+(?:[-']\w+)*|[^\w\s]", sentence)

    # Initialize all tokens with the default Outside tag
    tags = ["O"] * len(tokens)

    # Track tokens that have already been annotated to avoid overlapping entities
    occupied = [False] * len(tokens)

    # Create the list of all variants
    all_variants = []

    for canonical_name, variants in aliases.items():
        for variant in variants:
            all_variants.append(variant)

    # Match longer names first
    # (e.g. "Mount Everest" before "Everest")
    all_variants = sorted(
        all_variants,
        key=lambda x: len(x.split()),
        reverse=True
    )

    for variant in all_variants:

        variant_tokens = re.findall(r"\w+(?:[-']\w+)*|[^\w\s]", variant)

        variant_lower = [t.lower() for t in variant_tokens]

        n = len(variant_tokens)

        # Compare each window of tokens with the current mountain name
        for i in range(len(tokens)-n+1):

            window = [t.lower() for t in tokens[i:i+n]]

            if window == variant_lower:

                # Skip overlapping matches
                if any(occupied[i:i+n]):
                    continue

                tags[i] = "B-MOUNTAIN"

                occupied[i] = True

                for j in range(1, n):

                    tags[i+j] = "I-MOUNTAIN"
                    occupied[i+j] = True

    return tokens, tags

Example sentence used to verify BIO annotation.

In [ ]:
sentence = "We climbed Mount Everest before visiting K2 and Broad Peak."

tokens, tags = create_bio(sentence, aliases)

for t, tag in zip(tokens, tags):
    print(f"{t:15} {tag}")

We              O
climbed         O
Mount           B-MOUNTAIN
Everest         I-MOUNTAIN
before          O
visiting        O
K2              B-MOUNTAIN
and             O
Broad           B-MOUNTAIN
Peak            I-MOUNTAIN
.               O


## Exporting the final dataset for model training.

Each generated sentence is converted into the final dataset format containing:

- the original sentence
- tokenized words
- corresponding BIO labels

The resulting dataset is stored as a Pandas DataFrame before being converted
to the Hugging Face `Dataset` format for model training.

In [ ]:
dataset = []

for sentence in sentences:

    tokens, tags = create_bio(sentence, aliases)

    dataset.append({
        "sentence": sentence,
        "tokens": tokens,
        "ner_tags": tags
    })

In [ ]:
df = pd.DataFrame(dataset)

df.head()

,sentence,tokens,ner_tags
0,Everest appeared above the morning clouds just...,"[Everest, appeared, above, the, morning, cloud...","[B-MOUNTAIN, O, O, O, O, O, O, O, O, O, O, O, ..."
1,We left the campsite before dawn to avoid hiki...,"[We, left, the, campsite, before, dawn, to, av...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ..."
2,K2 has challenged experienced climbers with it...,"[K2, has, challenged, experienced, climbers, w...","[B-MOUNTAIN, O, O, O, O, O, O, O, O, O, O, O, ..."
3,The trail wound through a quiet forest filled ...,"[The, trail, wound, through, a, quiet, forest,...","[O, O, O, O, O, O, O, O, O, O, O, O, O]"
4,Tourists often compare the views from Matterho...,"[Tourists, often, compare, the, views, from, M...","[O, O, O, O, O, O, B-MOUNTAIN, O, O, O, O, O, ..."


Display one example to verify the generated annotations.

In [ ]:
print(df.iloc[0]["sentence"])
print()
print(df.iloc[0]["tokens"])
print()
print(df.iloc[0]["ner_tags"])

Everest appeared above the morning clouds just as the first rays of sunlight reached the valley.

['Everest', 'appeared', 'above', 'the', 'morning', 'clouds', 'just', 'as', 'the', 'first', 'rays', 'of', 'sunlight', 'reached', 'the', 'valley', '.']

['B-MOUNTAIN', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


### Dataset statistics

To better understand the generated corpus, several basic statistics are
calculated, including the total number of tokens and the proportion of
entity tokens.

In [ ]:
total_tokens = sum(len(tokens) for tokens in df["tokens"])

entity_tokens = sum(
    sum(label != "O" for label in labels)
    for labels in df["ner_tags"]
)

print(f"Total tokens: {total_tokens}")
print(f"Entity tokens: {entity_tokens}")
print(f"Entity ratio: {entity_tokens / total_tokens:.2%}")

Total tokens: 4203
Entity tokens: 268
Entity ratio: 6.38%


### Train, validation and test split

The dataset is converted to the Hugging Face `Dataset` format and randomly
split into training, validation and test subsets using a fixed random seed
to ensure reproducibility.

In [ ]:
hf_dataset = Dataset.from_pandas(df)

hf_dataset

Dataset({
    features: ['sentence', 'tokens', 'ner_tags'],
    num_rows: 398
})

Split the dataset into training (80%) and temporary (20%) subsets.

In [ ]:
dataset = hf_dataset.train_test_split(
    test_size=0.2,
    seed=42
)

Split the remaining 20% equally into validation and test sets.

In [ ]:
test_valid = dataset["test"].train_test_split(
    test_size=0.5,
    seed=42
)

dataset = {
    "train": dataset["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
}

Display the number of samples in each split.

In [ ]:
print(len(dataset["train"]))
print(len(dataset["validation"]))
print(len(dataset["test"]))

318
40
40


## Encoding BIO labels

Transformer models require numerical labels instead of string values.
Therefore, the BIO tags are mapped to integer identifiers before training.

Label mapping:

- O -> 0
- B-MOUNTAIN -> 1
- I-MOUNTAIN -> 2

In [ ]:
label_list = [
    "O",
    "B-MOUNTAIN",
    "I-MOUNTAIN"
]

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print(label2id)

{'O': 0, 'B-MOUNTAIN': 1, 'I-MOUNTAIN': 2}


In [ ]:
def encode_labels(example):

    example["labels"] = [
        label2id[label]
        for label in example["ner_tags"]
    ]

    return example

Convert the dataset splits into a Hugging Face DatasetDict, so the same preprocessing can be applied to every split.

In [ ]:
dataset = DatasetDict(dataset)

dataset = dataset.map(encode_labels)

Display one encoded training example.

In [ ]:
example = dataset["train"][14]

In [ ]:
print(example["sentence"])
print()
print(example["tokens"])
print()
print(example["labels"])

Hikers often compare the technical demands of K2 and Broad Peak.

['Hikers', 'often', 'compare', 'the', 'technical', 'demands', 'of', 'K2', 'and', 'Broad', 'Peak', '.']

[0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 2, 0]
